In [1]:
import numpy as np
import random
from copy import deepcopy

class PermutationSearcher:
    def __init__(self, L_half=6, P=768):
        self.L_half = L_half
        self.P = P
        self.identity = tuple(range(P))
        # 基準となる巡回置換 sigma (0, 1, ..., P-1)
        self.sigma = tuple((i + 1) % P for i in range(P))
        
    def multiply(self, p1, p2):
        """置換の積 p1 * p2"""
        return tuple(p1[p2[i]] for i in range(self.P))

    def invert(self, p):
        """置換の逆元 p^-1"""
        res = [0] * self.P
        for i, val in enumerate(p):
            res[val] = i
        return tuple(res)

    def get_cyclic_shift(self, k):
        """中心化群 C(sigma) の要素（巡回シフト）を生成"""
        k %= self.P
        return tuple((i + k) % self.P for i in range(self.P))

    def get_random_swap(self):
        """中心化群に属さない「欠陥」としてのランダムな互換を生成"""
        p = list(range(self.P))
        idx1, idx2 = random.sample(range(self.P), 2)
        p[idx1], p[idx2] = p[idx2], p[idx1]
        return tuple(p)

    def is_commuting(self, p1, p2):
        """可換性チェック [p1, p2] == 0"""
        return self.multiply(p1, p2) == self.multiply(p2, p1)

    def calculate_cycle_cost(self, F, G):
        """
        条件Cの簡易評価: 長さ4および6のサイクルの種をカウント
        (代数的閉路条件: F_i * G_j * inv(F_i') * inv(G_j') == Identity)
        """
        cost = 0
        # 長さ4のサイクルチェック (i, j, i', j')
        for i in range(self.L_half):
            for j in range(self.L_half):
                for ip in range(i + 1, self.L_half):
                    for jp in range(j + 1, self.L_half):
                        # パス積: F_i * G_j * inv(F_ip) * inv(G_jp)
                        prod = self.multiply(self.multiply(F[i], G[j]), 
                                             self.multiply(self.invert(F[ip]), self.invert(G[jp])))
                        if prod == self.identity:
                            cost += 100 # 強力なペナルティ
        return cost

    def solve(self, max_iter=1000):
        # 1. 初期化 (すべて巡回シフト = 条件Aを一旦満たす)
        F = [self.get_cyclic_shift(random.randint(0, self.P-1)) for _ in range(self.L_half)]
        G = [self.get_cyclic_shift(random.randint(0, self.P-1)) for _ in range(self.L_half)]
        
        # 2. 条件Bの注入 (0,3) と (1,2) を非可換にする
        # 巡回シフトの片方をランダム置換（互換）に置き換える
        F[0] = self.get_random_swap()
        G[2] = self.get_random_swap()
        
        # 条件Bの確認
        if self.is_commuting(F[0], G[3]) or self.is_commuting(F[1], G[2]):
            return self.solve() # 再試行

        current_cost = self.calculate_cycle_cost(F, G)
        
        # 3. 局所探索 (Hill Climbing)
        for it in range(max_iter):
            if current_cost == 0: break
            
            # ランダムに一つ選んで変更
            target_list = random.choice([F, G])
            idx = random.randint(0, self.L_half - 1)
            old_val = target_list[idx]
            
            # 属性（C(sigma)内か外か）を維持して変更
            if (target_list is F and idx in [0]) or (target_list is G and idx in [2]):
                target_list[idx] = self.get_random_swap()
            else:
                target_list[idx] = self.get_cyclic_shift(random.randint(0, self.P-1))
            
            # 条件A, Bの維持チェック
            valid_structure = True
            for i in range(self.L_half):
                for j in range(self.L_half):
                    if (i, j) in [(0, 3), (1, 2)]:
                        if self.is_commuting(F[i], G[j]): valid_structure = False
                    else:
                        if not self.is_commuting(F[i], G[j]): valid_structure = False
            
            if not valid_structure:
                target_list[idx] = old_val # 差し戻し
                continue

            new_cost = self.calculate_cycle_cost(F, G)
            if new_cost < current_cost:
                current_cost = new_cost
                print(f"Iteration {it}: Cost reduced to {current_cost}")
            else:
                target_list[idx] = old_val
                
        return F, G

if __name__ == "__main__":
    searcher = PermutationSearcher(L_half=6, P=768)
    final_F, final_G = searcher.solve()
    print("\nSearch complete.")
    print(f"Final F0 (defect example): {final_F[0][:10]}...") # 冒頭のみ表示


Search complete.
Final F0 (defect example): (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)...
